# Pickleball Positioning Coach — MVP Pipeline Demo

**Goal**: A positioning-and-movement coach for beginner/intermediate doubles, designed around a single phone mounted in an elevated back corner.

**Pipeline**:
```
Frame → Pose Estimation → Court Homography → Feature Engineering → Rubric Matching → LLM Translation → Coaching Feedback
```

**Design principles**:
- No ball tracking, no depth sensor — only 2D pose keypoints + homography
- The **rubric is the diagnostic engine**; the LLM only translates, never invents diagnoses
- An **abstention layer** blocks feedback when confidence is too low to be useful

**Court reference frame** (all positions in feet):
- Width: 20 ft (x-axis, 0 = left sideline)
- Length: 44 ft (y-axis, 0 = near baseline, 44 = far baseline)
- Net at y = 22; NVZ (kitchen) lines at y = 15 and y = 29

## Cell 1 — Install & Imports

In [ ]:
%pip install -q mediapipe opencv-python-headless openai numpy matplotlib Pillow python-dotenv

In [ ]:
from __future__ import annotations

import os
import json
import dataclasses
from dataclasses import dataclass, field
from typing import Optional

import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from dotenv import load_dotenv
import openai

load_dotenv()

# ── Court constants (feet) ─────────────────────────────────────────────────────
COURT_WIDTH_FT  = 20.0   # x: 0 = left sideline, 20 = right sideline
COURT_LENGTH_FT = 44.0   # y: 0 = near baseline, 44 = far baseline
NET_Y_FT        = 22.0   # net
KITCHEN_NEAR_FT = 15.0   # NVZ line on the near side
KITCHEN_FAR_FT  = 29.0   # NVZ line on the far side

# Analysing team on the NEAR side (y in 0–22).
# "At the kitchen" = y close to KITCHEN_NEAR_FT (15 ft).
TEAM_BASELINE_Y = 0.0
TEAM_KITCHEN_Y  = KITCHEN_NEAR_FT

# mediapipe 0.10.14+ removed the mp.solutions lazy-loader;
# the underlying modules still exist under mediapipe.python.solutions.
try:
    mp_pose_mod    = mp.solutions.pose
    mp_drawing_mod = mp.solutions.drawing_utils
except AttributeError:
    from mediapipe.python.solutions import pose           as mp_pose_mod
    from mediapipe.python.solutions import drawing_utils  as mp_drawing_mod

print("Imports OK — MediaPipe", mp.__version__)

## Cell 2 — Pose Estimation

Run MediaPipe Pose on a frame and extract the two players' foot positions (ankle midpoints) and an overall confidence score.

The detector returns a `PoseResult` per player. The **foot pixel** is the average of left/right ankle landmarks — the lowest reliably-visible point, anchoring the player to the ground plane for homography projection.

In [ ]:
@dataclass
class PoseResult:
    player_id: int
    foot_pixel: tuple[float, float]       # (x, y) in image pixels
    hip_pixel: tuple[float, float]        # used for posture checks
    confidence: float                     # min visibility of key joints
    raw_landmarks: object = field(default=None, repr=False)


def detect_poses(frame_bgr: np.ndarray) -> list[PoseResult]:
    """
    Run MediaPipe on the full frame.  Returns up to 2 PoseResults (one per
    detected person region).

    MediaPipe single-person Pose is run on the left and right halves of the
    frame as a cheap two-player split.  This is intentionally naive — a real
    system would use a person detector or multi-person MediaPipe.
    """
    h, w = frame_bgr.shape[:2]
    results: list[PoseResult] = []

    # Split frame into left / right halves (player 0 and player 1)
    halves = [
        (0, frame_bgr[:, : w // 2]),
        (1, frame_bgr[:, w // 2 :]),
    ]

    with mp_pose_mod.Pose(
        static_image_mode=True,
        model_complexity=1,
        min_detection_confidence=0.3,
    ) as pose:
        for player_id, half in halves:
            rgb = cv2.cvtColor(half, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)
            if res.pose_landmarks is None:
                continue

            lm = res.pose_landmarks.landmark
            hh, hw = half.shape[:2]

            # Landmark indices for key joints
            LEFT_HIP, RIGHT_HIP     = 23, 24
            LEFT_ANKLE, RIGHT_ANKLE = 27, 28

            ankle_vis = min(lm[LEFT_ANKLE].visibility, lm[RIGHT_ANKLE].visibility)
            hip_vis   = min(lm[LEFT_HIP].visibility,   lm[RIGHT_HIP].visibility)
            confidence = min(ankle_vis, hip_vis)

            # Foot pixel = midpoint of both ankles, in original frame coords
            foot_x_local = (lm[LEFT_ANKLE].x + lm[RIGHT_ANKLE].x) / 2 * hw
            foot_y_local = (lm[LEFT_ANKLE].y + lm[RIGHT_ANKLE].y) / 2 * hh
            # Offset right-half detections back to full-frame coords
            x_offset = (w // 2) * player_id
            foot_pixel = (foot_x_local + x_offset, foot_y_local)

            hip_x_local = (lm[LEFT_HIP].x + lm[RIGHT_HIP].x) / 2 * hw
            hip_y_local = (lm[LEFT_HIP].y + lm[RIGHT_HIP].y) / 2 * hh
            hip_pixel = (hip_x_local + x_offset, hip_y_local)

            results.append(
                PoseResult(
                    player_id=player_id,
                    foot_pixel=foot_pixel,
                    hip_pixel=hip_pixel,
                    confidence=float(confidence),
                    raw_landmarks=res.pose_landmarks,
                )
            )

    return results


def visualise_poses(frame_bgr: np.ndarray, poses: list[PoseResult]) -> None:
    vis = frame_bgr.copy()
    colors = [(0, 200, 255), (255, 130, 0)]
    for p in poses:
        cx, cy = int(p.foot_pixel[0]), int(p.foot_pixel[1])
        cv2.circle(vis, (cx, cy), 8, colors[p.player_id], -1)
        cv2.putText(
            vis,
            f"P{p.player_id} conf={p.confidence:.2f}",
            (cx + 10, cy - 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[p.player_id], 1,
        )
    plt.figure(figsize=(8, 4))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title("Pose foot detections")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


print("Pose detection functions defined.")

## Cell 3 — Court Homography

Map 4 pixel corners of the court (visible from the back-corner camera) to their known real-world positions in feet.  `cv2.findHomography` gives us the 3×3 matrix H; `warp_to_court` projects any pixel into court-space coordinates.

**Corner labeling convention** (for a back-left camera angle):

| Corner | Pixel label | Court (ft) |
|--------|-------------|------------|
| Near-left baseline | `src[0]` | (0, 0) |
| Near-right baseline | `src[1]` | (20, 0) |
| Far-right baseline | `src[2]` | (20, 44) |
| Far-left baseline | `src[3]` | (0, 44) |

In production these are clicked interactively; in this demo they're hardcoded from the synthetic frame.

In [ ]:
@dataclass
class Homography:
    H: np.ndarray                   # 3×3 matrix (pixel → court feet)
    reprojection_error: float       # mean pixel error on the 4 corners


def compute_homography(
    src_pixels: np.ndarray,         # shape (4, 2)  — pixel corners
    dst_court_ft: np.ndarray,       # shape (4, 2)  — court corners in feet
) -> Homography:
    H, _ = cv2.findHomography(src_pixels.astype(np.float32),
                               dst_court_ft.astype(np.float32))

    # Reprojection error on the 4 calibration points
    pts_h = np.hstack([src_pixels, np.ones((4, 1))]).astype(np.float64)
    projected = (H @ pts_h.T).T
    projected /= projected[:, [2]]
    error = float(np.mean(np.linalg.norm(projected[:, :2] - dst_court_ft, axis=1)))
    return Homography(H=H, reprojection_error=error)


def warp_to_court(pixel_xy: tuple[float, float], hom: Homography) -> tuple[float, float]:
    """Project a single image pixel to court coordinates (feet)."""
    p = np.array([pixel_xy[0], pixel_xy[1], 1.0], dtype=np.float64)
    q = hom.H @ p
    q /= q[2]
    return float(q[0]), float(q[1])


def visualise_birdseye(
    court_positions: list[tuple[float, float]],
    labels: list[str],
    title: str = "Bird's-eye court view",
) -> None:
    fig, ax = plt.subplots(figsize=(4, 7))
    # Court outline
    ax.add_patch(mpatches.Rectangle((0, 0), COURT_WIDTH_FT, COURT_LENGTH_FT,
                                     linewidth=2, edgecolor="white", facecolor="#3a7d2c"))
    # Net
    ax.plot([0, COURT_WIDTH_FT], [NET_Y_FT, NET_Y_FT], "w--", lw=1.5, label="Net")
    # Kitchen lines
    for ky in (KITCHEN_NEAR_FT, KITCHEN_FAR_FT):
        ax.plot([0, COURT_WIDTH_FT], [ky, ky], color="#aad4ff", lw=1.2)
    # Centreline
    ax.plot([COURT_WIDTH_FT / 2, COURT_WIDTH_FT / 2], [0, COURT_LENGTH_FT],
            color="white", lw=0.8, alpha=0.5)

    colors = ["#FFD700", "#FF6347"]
    for pos, lbl, col in zip(court_positions, labels, colors):
        ax.scatter(*pos, s=180, color=col, zorder=5, edgecolors="white", linewidths=1)
        ax.text(pos[0] + 0.4, pos[1] + 0.4, lbl, color=col, fontsize=9, fontweight="bold")

    ax.set_xlim(-1, COURT_WIDTH_FT + 1)
    ax.set_ylim(-1, COURT_LENGTH_FT + 1)
    ax.set_aspect("equal")
    ax.set_facecolor("#3a7d2c")
    ax.set_xlabel("Width (ft)")
    ax.set_ylabel("Length (ft)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# ── Demo: hardcoded corner pixels for the synthetic frame (defined in Cell 4) ──
# These match the perspective trapezoid drawn by make_synthetic_frame().
DEMO_SRC_PIXELS = np.array([
    [ 80, 440],   # near-left baseline
    [560, 440],   # near-right baseline
    [500,  60],   # far-right baseline
    [140,  60],   # far-left baseline
], dtype=np.float32)

DEMO_DST_COURT = np.array([
    [ 0,  0],
    [20,  0],
    [20, 44],
    [ 0, 44],
], dtype=np.float32)

demo_hom = compute_homography(DEMO_SRC_PIXELS, DEMO_DST_COURT)
print(f"Homography computed — reprojection error: {demo_hom.reprojection_error:.4f} ft")

## Cell 4 — Synthetic Frame Generator

Produces a perspective-correct court image with two stick-figure players at any court-space position.  This makes the demo fully self-contained — no real video required.

In [ ]:
def make_synthetic_frame(
    player0_court: tuple[float, float] = (5.0, 2.5),   # near-side, left
    player1_court: tuple[float, float] = (15.0, 2.5),  # near-side, right
    frame_w: int = 640,
    frame_h: int = 480,
) -> tuple[np.ndarray, list[tuple[float, float]]]:
    """
    Render a perspective view of a pickleball court with two players.

    The inverse homography (court→pixel) is computed from DEMO_DST_COURT /
    DEMO_SRC_PIXELS, so the ground-truth foot pixels are exact.

    Returns:
        frame_bgr  — uint8 BGR image
        foot_pixels — [(x0,y0), (x1,y1)] in pixel space
    """
    H_inv, _ = cv2.findHomography(DEMO_DST_COURT, DEMO_SRC_PIXELS)

    def court_to_pixel(cx: float, cy: float) -> tuple[int, int]:
        p = H_inv @ np.array([cx, cy, 1.0])
        p /= p[2]
        return int(p[0]), int(p[1])

    # Background
    frame = np.full((frame_h, frame_w, 3), (50, 120, 50), dtype=np.uint8)

    # Draw court outline
    corners = [court_to_pixel(x, y) for x, y in [(0,0),(20,0),(20,44),(0,44)]]
    cv2.polylines(frame, [np.array(corners, np.int32)], isClosed=True,
                  color=(220, 220, 220), thickness=2)

    # Net
    net_l, net_r = court_to_pixel(0, NET_Y_FT), court_to_pixel(20, NET_Y_FT)
    cv2.line(frame, net_l, net_r, (180, 180, 220), 2)

    # Kitchen lines
    for ky in (KITCHEN_NEAR_FT, KITCHEN_FAR_FT):
        kl, kr = court_to_pixel(0, ky), court_to_pixel(20, ky)
        cv2.line(frame, kl, kr, (150, 200, 255), 1)

    # Centreline
    cl, cr = court_to_pixel(10, 0), court_to_pixel(10, 44)
    cv2.line(frame, cl, cr, (180, 180, 180), 1)

    # Draw players as stick figures
    foot_pixels: list[tuple[float, float]] = []
    player_colors = [(0, 200, 255), (255, 130, 0)]

    for court_pos, col in zip([player0_court, player1_court], player_colors):
        fx, fy = court_to_pixel(*court_pos)
        foot_pixels.append((float(fx), float(fy)))

        # Perspective scale: players look smaller near the net/far end
        # Rough scale: bigger near camera (y≈0), smaller far away (y≈22)
        scale = max(0.4, 1.0 - court_pos[1] / 35.0)
        sh = int(70 * scale)   # figure height in pixels

        # Stick figure: ankles → hips → head, arms
        ankle  = (fx, fy)
        hip    = (fx, fy - int(sh * 0.45))
        head   = (fx, fy - int(sh * 0.85))
        lshldr = (fx - int(sh * 0.25), fy - int(sh * 0.55))
        rshldr = (fx + int(sh * 0.25), fy - int(sh * 0.55))
        lelbow = (fx - int(sh * 0.30), fy - int(sh * 0.30))
        relbow = (fx + int(sh * 0.30), fy - int(sh * 0.30))
        lknee  = (fx - int(sh * 0.10), fy - int(sh * 0.20))
        rknee  = (fx + int(sh * 0.10), fy - int(sh * 0.20))

        for seg in [
            (ankle, lknee), (ankle, rknee),     # legs
            (lknee, hip),   (rknee, hip),
            (hip, lshldr),  (hip, rshldr),       # torso
            (lshldr, lelbow), (rshldr, relbow),  # arms
        ]:
            cv2.line(frame, seg[0], seg[1], col, max(1, int(3 * scale)))
        cv2.circle(frame, head, int(9 * scale), col, -1)

    return frame, foot_pixels


# ── Scenario catalogue ─────────────────────────────────────────────────────────
SCENARIOS: dict[str, dict] = {
    "both_at_kitchen": {
        "p0": (4.0, 15.0),
        "p1": (16.0, 15.0),
        "desc": "Both players correctly positioned at the NVZ line",
    },
    "one_up_one_back": {
        "p0": (5.0, 14.0),
        "p1": (14.0, 3.0),
        "desc": "Classic split — one at kitchen, partner stuck at baseline",
    },
    "both_baseline": {
        "p0": (5.0, 2.0),
        "p1": (15.0, 2.0),
        "desc": "Both players camping at baseline, ceding the kitchen",
    },
    "middle_exposed": {
        "p0": (1.5, 14.5),
        "p1": (18.5, 14.5),
        "desc": "Both at kitchen but stacked too wide, big gap down the middle",
    },
    "stacked_same_side": {
        "p0": (3.0, 13.5),
        "p1": (6.0, 14.0),
        "desc": "Both players crowded to the left — right side wide open",
    },
}

# Quick preview
scenario_name = "one_up_one_back"
sc = SCENARIOS[scenario_name]
frame_bgr, ground_truth_feet = make_synthetic_frame(sc["p0"], sc["p1"])

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
plt.title(f"Synthetic frame — '{scenario_name}'\n{sc['desc']}")
plt.axis("off")
plt.tight_layout()
plt.show()
print("Ground-truth foot pixels:", ground_truth_feet)

## Cell 5 — Feature Engineering

Given two foot positions in court-space (feet), compute the positioning metrics that the rubric will evaluate.

All features are interpretable scalars or categories — no raw pixel values cross this boundary.

In [ ]:
@dataclass
class PositioningFeatures:
    # Raw court positions
    p0_court: tuple[float, float]
    p1_court: tuple[float, float]

    # Derived metrics
    p0_kitchen_dist: float      # ft from NVZ line (0 = on line, + = behind)
    p1_kitchen_dist: float
    kitchen_dist_spread: float  # |p0 - p1| kitchen distances — measures one-up/one-back
    partner_gap: float          # lateral (x-axis) distance between players
    court_balance: float        # 0 = both far left, 1 = both far right, 0.5 = centered split
    court_coverage: float       # fraction of court width spanned (0–1)
    p0_zone: str                # "kitchen" | "mid" | "baseline"
    p1_zone: str

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)


def _zone(kitchen_dist: float) -> str:
    if kitchen_dist <= 2.5:
        return "kitchen"
    if kitchen_dist <= 10.0:
        return "mid"
    return "baseline"


def extract_features(
    p0_court: tuple[float, float],
    p1_court: tuple[float, float],
) -> PositioningFeatures:
    """Compute all positioning metrics from two court-space foot positions."""
    p0x, p0y = p0_court
    p1x, p1y = p1_court

    # Kitchen distance: how far back from the NVZ line (clipped to ≥0)
    p0_kd = max(0.0, TEAM_KITCHEN_Y - p0y)
    p1_kd = max(0.0, TEAM_KITCHEN_Y - p1y)

    spread        = abs(p0_kd - p1_kd)
    lateral_gap   = abs(p0x - p1x)

    mid_x         = (p0x + p1x) / 2
    court_balance = mid_x / COURT_WIDTH_FT          # 0.5 = centred

    leftmost  = min(p0x, p1x)
    rightmost = max(p0x, p1x)
    coverage  = (rightmost - leftmost) / COURT_WIDTH_FT

    return PositioningFeatures(
        p0_court=p0_court,
        p1_court=p1_court,
        p0_kitchen_dist=round(p0_kd, 2),
        p1_kitchen_dist=round(p1_kd, 2),
        kitchen_dist_spread=round(spread, 2),
        partner_gap=round(lateral_gap, 2),
        court_balance=round(court_balance, 3),
        court_coverage=round(coverage, 3),
        p0_zone=_zone(p0_kd),
        p1_zone=_zone(p1_kd),
    )


# Quick smoke test
test_feat = extract_features((5.0, 14.0), (14.0, 3.0))
print("Feature engineering smoke-test (one_up_one_back):")
for k, v in test_feat.to_dict().items():
    if not k.endswith("_court"):
        print(f"  {k:28s} {v}")

## Cell 6 — Explicit Rubric

The rubric encodes the coaching logic as explicit threshold rules.  **The LLM never diagnoses — it only translates these matched rules into plain English.**

Each rule has:
- `condition`: a callable that takes a `PositioningFeatures` and returns `bool`
- `feedback_template`: the canonical advice the LLM will rephrase
- `priority`: `high` | `medium` | `low`
- `category`: `positioning` | `movement` | `teamwork`

In [ ]:
@dataclass
class RubricRule:
    id: str
    condition: object           # Callable[[PositioningFeatures], bool]
    feedback_template: str
    priority: str               # "high" | "medium" | "low"
    category: str               # "positioning" | "movement" | "teamwork"


@dataclass
class RubricMatch:
    rule_id: str
    priority: str
    category: str
    feedback_template: str
    metric_snapshot: dict       # relevant feature values at time of match


# ─────────────────────────────────────────────────────────────────────────────
# The full rubric — 8 rules covering the most common beginner positioning errors
# ─────────────────────────────────────────────────────────────────────────────
RUBRIC: list[RubricRule] = [
    RubricRule(
        id="both_at_kitchen",
        condition=lambda f: f.p0_kitchen_dist <= 2.5 and f.p1_kitchen_dist <= 2.5,
        feedback_template=(
            "Both players are at the Non-Volley Zone line — this is the ideal net position. "
            "Hold this spot and be ready to reset dinks."
        ),
        priority="high",
        category="positioning",
    ),
    RubricRule(
        id="one_up_one_back",
        condition=lambda f: f.kitchen_dist_spread >= 5.0,
        feedback_template=(
            "One player is at the kitchen while the other is still near the baseline — "
            "this split leaves a wide gap. The back player should advance after a safe third shot."
        ),
        priority="high",
        category="movement",
    ),
    RubricRule(
        id="both_baseline",
        condition=lambda f: f.p0_kitchen_dist >= 10.0 and f.p1_kitchen_dist >= 10.0,
        feedback_template=(
            "Both players are camped near the baseline. You're giving away the kitchen — "
            "look for an opportunity to both advance to the NVZ together."
        ),
        priority="high",
        category="positioning",
    ),
    RubricRule(
        id="middle_exposed",
        condition=lambda f: (
            f.partner_gap >= 9.0
            and f.p0_kitchen_dist <= 3.5
            and f.p1_kitchen_dist <= 3.5
        ),
        feedback_template=(
            "Both players are at the kitchen but spread too wide. "
            "The middle gap is a common target — shift inward 1-2 feet to close it."
        ),
        priority="high",
        category="teamwork",
    ),
    RubricRule(
        id="stacked_same_side",
        condition=lambda f: f.court_coverage <= 0.35 and f.court_balance < 0.35,
        feedback_template=(
            "Both players are crowded to the left side. "
            "Spread out to cover the full court width — your opponents can drive to the open right side."
        ),
        priority="medium",
        category="teamwork",
    ),
    RubricRule(
        id="stacked_same_side_right",
        condition=lambda f: f.court_coverage <= 0.35 and f.court_balance > 0.65,
        feedback_template=(
            "Both players are crowded to the right side. "
            "Spread out to cover the full court width — the open left side is exposed."
        ),
        priority="medium",
        category="teamwork",
    ),
    RubricRule(
        id="mid_court_no_man",
        condition=lambda f: (
            f.p0_zone == "mid" or f.p1_zone == "mid"
        ) and f.kitchen_dist_spread <= 2.0,
        feedback_template=(
            "One or both players are stuck in no-man's-land (the mid-court transition zone). "
            "This is a vulnerable position — either advance fully to the kitchen or retreat to the baseline."
        ),
        priority="medium",
        category="movement",
    ),
    RubricRule(
        id="good_court_coverage",
        condition=lambda f: (
            f.court_coverage >= 0.55
            and f.court_balance >= 0.35
            and f.court_balance <= 0.65
        ),
        feedback_template=(
            "Good lateral spread — you're covering the court width well. "
            "Now focus on forward positioning: both players should work toward the kitchen."
        ),
        priority="low",
        category="teamwork",
    ),
]


def evaluate_rubric(features: PositioningFeatures) -> list[RubricMatch]:
    """Return all triggered rubric rules, sorted by priority."""
    priority_order = {"high": 0, "medium": 1, "low": 2}
    matches: list[RubricMatch] = []

    for rule in RUBRIC:
        try:
            triggered = rule.condition(features)
        except Exception:
            continue
        if triggered:
            snapshot = {
                "p0_kitchen_dist": features.p0_kitchen_dist,
                "p1_kitchen_dist": features.p1_kitchen_dist,
                "kitchen_dist_spread": features.kitchen_dist_spread,
                "partner_gap": features.partner_gap,
                "court_balance": features.court_balance,
                "court_coverage": features.court_coverage,
                "p0_zone": features.p0_zone,
                "p1_zone": features.p1_zone,
            }
            matches.append(RubricMatch(
                rule_id=rule.id,
                priority=rule.priority,
                category=rule.category,
                feedback_template=rule.feedback_template,
                metric_snapshot=snapshot,
            ))

    matches.sort(key=lambda m: priority_order.get(m.priority, 99))
    return matches


# Smoke test
matches = evaluate_rubric(test_feat)
print(f"Rubric matched {len(matches)} rule(s) for 'one_up_one_back':")
for m in matches:
    print(f"  [{m.priority:6s}] {m.rule_id}")

## Cell 7 — Abstention Layer

Before calling the LLM, we gate on data quality.  If any check fails, the relevant feedback is suppressed and an honest "I can't tell" note is returned instead.

This is the **differentiator**: confident silence is more trustworthy than fluent hallucination.

In [ ]:
CONFIDENCE_THRESHOLD   = 0.50   # min acceptable MediaPipe visibility score
REPROJECTION_THRESHOLD = 1.5    # max acceptable homography reprojection error (ft)

# When running on synthetic data, we inject synthetic pose confidence
SYNTHETIC_CONFIDENCE = 0.95


@dataclass
class AbstentionReport:
    should_abstain_all: bool            # True → no LLM call at all
    abstain_reasons: list[str]          # human-readable reasons
    uncertain_players: list[int]        # player IDs with low pose confidence
    suppressed_categories: list[str]    # rubric categories to skip


def check_abstention(
    poses: list[PoseResult],
    hom: Homography,
    features: Optional[PositioningFeatures] = None,
) -> AbstentionReport:
    reasons: list[str] = []
    uncertain_players: list[int] = []
    suppressed_cats: list[str] = []

    # 1. Homography quality
    if hom.reprojection_error > REPROJECTION_THRESHOLD:
        reasons.append(
            f"Court calibration uncertain (reprojection error {hom.reprojection_error:.2f} ft > "
            f"{REPROJECTION_THRESHOLD} ft threshold). Position feedback suppressed."
        )
        suppressed_cats.extend(["positioning", "movement", "teamwork"])

    # 2. Player count
    if len(poses) < 2:
        reasons.append(
            f"Only {len(poses)} player(s) detected (need 2 for team feedback). "
            "Teamwork feedback suppressed."
        )
        suppressed_cats.append("teamwork")

    # 3. Individual pose confidence
    for p in poses:
        if p.confidence < CONFIDENCE_THRESHOLD:
            reasons.append(
                f"Player {p.player_id} pose confidence too low ({p.confidence:.2f} < "
                f"{CONFIDENCE_THRESHOLD}). Feedback for this player suppressed."
            )
            uncertain_players.append(p.player_id)

    # 4. Out-of-bounds positions (homography extrapolation artefacts)
    if features is not None:
        for pid, pos in [(0, features.p0_court), (1, features.p1_court)]:
            if not (0 <= pos[0] <= COURT_WIDTH_FT and 0 <= pos[1] <= NET_Y_FT):
                reasons.append(
                    f"Player {pid} projected outside court bounds {pos} — "
                    "position feedback for this player suppressed."
                )
                uncertain_players.append(pid)

    # Abstain entirely only if calibration is bad or no players at all
    abstain_all = (
        hom.reprojection_error > REPROJECTION_THRESHOLD
        or len(poses) == 0
    )

    return AbstentionReport(
        should_abstain_all=abstain_all,
        abstain_reasons=reasons,
        uncertain_players=list(set(uncertain_players)),
        suppressed_categories=list(set(suppressed_cats)),
    )


# Smoke test — good calibration, synthetic confidence
synthetic_poses = [
    PoseResult(0, (200.0, 380.0), (200.0, 340.0), SYNTHETIC_CONFIDENCE),
    PoseResult(1, (450.0, 380.0), (450.0, 340.0), SYNTHETIC_CONFIDENCE),
]
report = check_abstention(synthetic_poses, demo_hom, test_feat)
print("Abstention report (clean data):")
print(f"  abstain_all          = {report.should_abstain_all}")
print(f"  uncertain_players    = {report.uncertain_players}")
print(f"  suppressed_cats      = {report.suppressed_categories}")
print(f"  reasons              = {report.abstain_reasons or ['none']}")

## Cell 8 — LLM Feedback

The LLM's role is **translation only** — convert the rubric matches into natural, contextual coaching language. It cannot add new diagnoses; all coaching logic lives in the rubric.

Key constraints baked into the system prompt:
1. Only comment on rules present in `rubric_matches`
2. Do not mention metrics or numbers the player wouldn't understand
3. If `abstain_reasons` is non-empty, acknowledge uncertainty honestly
4. Limit to 2-3 actionable tips — brevity is a feature

`OPENAI_API_KEY` is read from the environment (put it in a `.env` file).

In [ ]:
SYSTEM_PROMPT = """
You are a concise pickleball positioning coach for beginner and intermediate doubles players.

You will receive:
- A JSON object with `rubric_matches`: a list of pre-diagnosed positioning issues (each with a `feedback_template` and `priority`)
- `abstain_reasons`: a list of things you must NOT comment on because the data was unreliable
- `scenario_description`: brief context about the situation

Your job:
1. Rephrase the feedback_templates into 2-3 warm, plain-English coaching tips (one sentence each).
2. Prioritise HIGH-priority matches first. Skip LOW-priority ones if there are already 2 HIGH/MEDIUM tips.
3. Do NOT invent new diagnoses beyond what is in rubric_matches.
4. Do NOT mention numbers, distances, or pixel coordinates.
5. If abstain_reasons is non-empty, start with one honest sentence acknowledging what you could not assess.
6. Keep the entire response under 120 words.
7. Use "you" (addressing both players), not "Player 0" or "Player 1".
""".strip()


def get_llm_feedback(
    rubric_matches: list[RubricMatch],
    abstention: AbstentionReport,
    scenario_description: str = "",
    model: str = "gpt-4o-mini",
    dry_run: bool = False,
) -> str:
    """
    Call the OpenAI Chat API to translate rubric matches into coaching tips.

    Set dry_run=True (or leave OPENAI_API_KEY unset) to see the prompt
    without making an API call.
    """
    # Filter out matches whose category is suppressed
    active_matches = [
        m for m in rubric_matches
        if m.category not in abstention.suppressed_categories
           and not any(pid in abstention.uncertain_players for pid in range(2))
           or m.priority == "high"  # always include high-priority if at least 1 player OK
    ]
    # Re-filter: if all players uncertain, suppress everything
    if len(abstention.uncertain_players) == 2:
        active_matches = []

    payload = {
        "rubric_matches": [
            {
                "rule_id": m.rule_id,
                "priority": m.priority,
                "category": m.category,
                "feedback_template": m.feedback_template,
            }
            for m in active_matches
        ],
        "abstain_reasons": abstention.abstain_reasons,
        "scenario_description": scenario_description,
    }

    user_content = json.dumps(payload, indent=2)

    if dry_run or not os.environ.get("OPENAI_API_KEY"):
        print("─── DRY RUN — LLM prompt (no API call) ───")
        print("SYSTEM:\n", SYSTEM_PROMPT[:300], "...\n")
        print("USER:\n", user_content)
        if not active_matches:
            return "(No active rubric matches — nothing to say.)"
        return (
            "[DRY RUN] LLM would rephrase the following templates:\n"
            + "\n".join(f"  • {m.feedback_template[:80]}…" for m in active_matches)
        )

    client = openai.OpenAI()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ],
        temperature=0.4,
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()


print("LLM feedback function defined.")
print(f"OPENAI_API_KEY present: {bool(os.environ.get('OPENAI_API_KEY'))}")

## Cell 9 — End-to-End Demo

Run the full pipeline on every scenario.  No real video needed — the synthetic frame generator provides the ground-truth foot pixels, bypassing MediaPipe (which wouldn't work on simple stick figures anyway).

In a real deployment the only change is: replace the `ground_truth_feet` line with the output of `detect_poses()` on a real frame.

In [ ]:
def run_pipeline(
    scenario_name: str,
    dry_run: bool = True,
    show_plots: bool = True,
) -> None:
    sc = SCENARIOS[scenario_name]
    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario_name}")
    print(f"  {sc['desc']}")
    print(f"{'='*60}")

    # ── Step 1: Generate synthetic frame ────────────────────────────────────
    frame_bgr, foot_pixels = make_synthetic_frame(sc["p0"], sc["p1"])

    # ── Step 2: Pose estimation (synthetic bypass) ───────────────────────────
    # On real video: poses = detect_poses(frame_bgr)
    poses = [
        PoseResult(0, foot_pixels[0], (foot_pixels[0][0], foot_pixels[0][1] - 40),
                   SYNTHETIC_CONFIDENCE),
        PoseResult(1, foot_pixels[1], (foot_pixels[1][0], foot_pixels[1][1] - 40),
                   SYNTHETIC_CONFIDENCE),
    ]
    print(f"\n[Pose] {len(poses)} players detected, "
          f"conf = [{poses[0].confidence:.2f}, {poses[1].confidence:.2f}]")

    # ── Step 3: Court homography ─────────────────────────────────────────────
    # On real video: supply clicked pixel corners; here we use demo_hom
    hom = demo_hom
    court_positions = [warp_to_court(p.foot_pixel, hom) for p in poses]
    print(f"[Hom]  Court positions: P0={court_positions[0]}, P1={court_positions[1]}")
    print(f"       Reprojection error: {hom.reprojection_error:.4f} ft")

    # ── Step 4: Feature engineering ─────────────────────────────────────────
    features = extract_features(court_positions[0], court_positions[1])
    print(f"\n[Feat] p0_kitchen_dist={features.p0_kitchen_dist} ft  "
          f"p1_kitchen_dist={features.p1_kitchen_dist} ft")
    print(f"       spread={features.kitchen_dist_spread} ft  "
          f"partner_gap={features.partner_gap} ft  "
          f"coverage={features.court_coverage:.0%}")
    print(f"       zones: P0={features.p0_zone}  P1={features.p1_zone}")

    # ── Step 5: Rubric evaluation ────────────────────────────────────────────
    matches = evaluate_rubric(features)
    print(f"\n[Rubric] {len(matches)} rule(s) matched:")
    for m in matches:
        print(f"  [{m.priority:6s}] {m.rule_id}")

    # ── Step 6: Abstention check ─────────────────────────────────────────────
    abstention = check_abstention(poses, hom, features)
    if abstention.abstain_reasons:
        print(f"\n[Abstain] {len(abstention.abstain_reasons)} flag(s):")
        for r in abstention.abstain_reasons:
            print(f"  ⚠ {r}")
    else:
        print("\n[Abstain] All checks passed — no abstentions.")

    if abstention.should_abstain_all:
        print("\n[LLM] Skipped — full abstention.")
        feedback = "(Feedback suppressed: data quality too low to coach reliably.)"
    else:
        # ── Step 7: LLM feedback ─────────────────────────────────────────────
        feedback = get_llm_feedback(
            matches, abstention,
            scenario_description=sc["desc"],
            dry_run=dry_run,
        )

    print(f"\n[Coaching Feedback]\n{feedback}")

    # ── Visualisations ───────────────────────────────────────────────────────
    if show_plots:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Left: camera frame
        axes[0].imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Camera view — '{scenario_name}'")
        axes[0].axis("off")
        for fp, col in zip(foot_pixels, ["#FFD700", "#FF6347"]):
            axes[0].plot(*fp, "o", color=col, markersize=10, mew=2,
                         mec="white", label=f"P{foot_pixels.index(fp)}")
        axes[0].legend(loc="upper right", fontsize=8)

        # Right: bird's-eye court
        ax = axes[1]
        ax.add_patch(mpatches.Rectangle((0, 0), COURT_WIDTH_FT, COURT_LENGTH_FT,
                                         lw=2, ec="white", fc="#3a7d2c"))
        ax.plot([0, COURT_WIDTH_FT], [NET_Y_FT, NET_Y_FT], "w--", lw=1.5)
        for ky in (KITCHEN_NEAR_FT, KITCHEN_FAR_FT):
            ax.plot([0, COURT_WIDTH_FT], [ky, ky], color="#aad4ff", lw=1.2)
        ax.plot([COURT_WIDTH_FT/2]*2, [0, COURT_LENGTH_FT], "w-", lw=0.8, alpha=0.5)

        # Annotate NVZ
        ax.text(10, KITCHEN_NEAR_FT - 0.8, "NVZ", ha="center", color="#aad4ff",
                fontsize=8)

        # Players
        for pos, pid, col in zip(court_positions, [0, 1], ["#FFD700", "#FF6347"]):
            ax.scatter(*pos, s=200, color=col, zorder=5, ec="white", lw=1)
            ax.text(pos[0]+0.4, pos[1]+0.5, f"P{pid}", color=col, fontsize=9, fw="bold")

        # Rubric match banners
        match_text = "\n".join(
            f"{'●' if m.priority=='high' else '○'} {m.rule_id}"
            for m in matches[:3]
        ) or "No issues"
        ax.text(0.5, -0.10, f"Rules: {match_text}", transform=ax.transAxes,
                ha="center", fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#222", alpha=0.8))

        ax.set_xlim(-1, COURT_WIDTH_FT + 1)
        ax.set_ylim(-1, COURT_LENGTH_FT + 1)
        ax.set_aspect("equal")
        ax.set_facecolor("#3a7d2c")
        ax.set_xlabel("Width (ft)")
        ax.set_ylabel("Length (ft)")
        ax.set_title("Bird's-eye court projection")
        plt.suptitle(f"Pipeline output: {scenario_name}", fontsize=11, y=1.01)
        plt.tight_layout()
        plt.show()


# ── Run all scenarios ──────────────────────────────────────────────────────────
for sname in SCENARIOS:
    run_pipeline(sname, dry_run=not bool(os.environ.get("OPENAI_API_KEY")))

## Bonus — Abstention Layer Demo

Shows the system correctly suppressing feedback when data quality is poor.

In [ ]:
print("─── Test 1: Low pose confidence (single player, low visibility) ───")
low_conf_poses = [
    PoseResult(0, (200.0, 380.0), (200.0, 340.0), confidence=0.28),   # below threshold
]
bad_report = check_abstention(low_conf_poses, demo_hom)
print(f"abstain_all={bad_report.should_abstain_all}  reasons={bad_report.abstain_reasons}")

print()
print("─── Test 2: Bad homography (e.g. camera angle changed) ───")
# Scrambled corners to simulate a poor calibration
bad_src = np.array([[10,10],[630,10],[400,470],[230,470]], dtype=np.float32)
bad_hom = compute_homography(bad_src, DEMO_DST_COURT)
print(f"Reprojection error: {bad_hom.reprojection_error:.2f} ft  "
      f"(threshold = {REPROJECTION_THRESHOLD} ft)")
bad_hom_report = check_abstention(synthetic_poses, bad_hom)
print(f"abstain_all={bad_hom_report.should_abstain_all}")
print(f"reasons: {bad_hom_report.abstain_reasons}")

print()
print("─── Test 3: Correct abstention in feedback pipeline ───")
feedback = get_llm_feedback(matches, bad_hom_report, dry_run=True)